# Exam-OCR LoRA training on Colab (T4/L4/A100)
Runtime → GPU. Data zip is built locally with `scripts/make_colab_bundle.py` and put in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE='/content/drive/MyDrive/exam-ocr'  # holds exam_ocr_data.zip; adapters are saved here

In [ ]:
!git clone https://github.com/alireza-taheriF/exam-ocr.git /content/exam-ocr
%cd /content/exam-ocr
!git checkout cursor/report-accuracy-graphs
!unzip -q -o $DRIVE/exam_ocr_data.zip -d .

In [ ]:
# Qwen3-VL needs a recent transformers; do NOT install requirements.txt (its surya pin conflicts).
!pip install -q -U 'transformers>=4.57' 'peft>=0.12' 'accelerate>=0.33' datasets pillow

In [ ]:
!python training/train_qwen_vl_lora.py \
  --train dataset/qwen_ft/train_mix_indomain.jsonl \
  --val dataset/qwen_ft/val_gold.jsonl \
  --output /content/adapter \
  --device cuda --epochs 2 --batch-size 4 --grad-accum 4 --max-long-side 768

In [ ]:
!mkdir -p $DRIVE/adapters && cp -r /content/adapter $DRIVE/adapters/lora-$(date +%Y%m%d-%H%M)
# Then locally: download the adapter into models/ and run eval.new_gold_report / eval.run_baseline.